In [4]:
%pip install -U langchain langchain-community langchain-groq langchain-chroma langchain-huggingface pypdf sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.9/382.9 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.6/739.6 kB 19.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.9/248.9 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.8/161.8 kB 8.3 MB/s eta 0:00:00
  Attempting uninstall: pypdf
    Found existing installation: pypdf 6.14.2
    Uninstalling pypdf-6.14.2:
      Successfully uninstalled pypdf-6.14.2
  Attempting uninstall: langgraph-sdk
    Found existing installation: langgraph-sdk 0.3.13
    Uninstalling langgraph-sdk-0.3.13:
      Successfully uninstalled langgraph-sdk-0.3.13
  Attempting uninstall: langgraph-checkpoint
    Found existing installation: langgraph-checkpoint 4.0

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

os.environ["GROQ_API_KEY"] = "API_KEY"

In [5]:
PDF_FILE_PATH = "/kaggle/input/datasets/saharsahebi/sample/sampleFile.pdf" 

print("1. Loading PDF...")
loader = PyPDFLoader(PDF_FILE_PATH)
docs = loader.load()
print(f"Loaded {len(docs)} pages.")

print("2. Chunking Text...")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)
print(f"Split into {len(splits)} chunks.")

print("3. Creating Vector Database...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("4. Initializing LLM and Chain...")
llm = ChatGroq(temperature=0, model_name="openai/gpt-oss-120b")

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert AI assistant analyzing a PDF document. Answer the user's question based ONLY on the provided context below. Do not guess or use outside knowledge.\n\nContext:\n{context}"),
    ("human", "{input}")
])

question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

print(" RAG Pipeline Ready!")

1. Loading PDF...
Loaded 73 pages.
2. Chunking Text...
Split into 99 chunks.
3. Creating Vector Database...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


4. Initializing LLM and Chain...
 RAG Pipeline Ready!


In [6]:
question = "Who are the main characters in this story, and what are their key personality traits?"
print(f" You: {question}")
print(" Thinking...\n")

response = rag_chain.invoke({"input": question})

print(" Assistant:")
print(response["answer"])

 You: Who are the main characters in this story, and what are their key personality traits?
 Thinking...

🤖 Assistant:
**Main characters (as they appear in the excerpt)**  

| Character | Role in the excerpt | Personality clues that the text gives |
|-----------|--------------------|----------------------------------------|
| **Jane Eyre** | The narrator; a former governess at Thornfield Hall who has just learned that she has inherited a large sum of money. | • She is **inquisitive** – she asks “Did Mr. Briggs write anything about Mr. Rochester?” and “Why did Mr. Briggs write to you?”  <br>• She is **thoughtful and deliberate** – “I thought carefully for a moment.”  <br>• She is **caring and responsible** – she immediately decides to write to her newly‑discovered cousins, Diana and Mary. |
| **St John** | The person delivering the news about the inheritance and the family connection. | • He is **informative and matter‑of‑fact**, delivering the facts about the uncle’s death and the £20,